# Time-Derivative Accuracy Demo

This notebook compares the analytic `solidfmm` time-derivative path against direct-sum reference formulas for jerk, snap, and crackle on a small particle set.

In [ ]:
import jax
import jax.numpy as jnp

from jaccpot import FastMultipoleMethod, FMMPreset


In [ ]:
def direct_sum_jerk(positions, masses, velocities, *, G=1.0, softening=1e-3):
    diff = positions[:, None, :] - positions[None, :, :]
    vdiff = velocities[:, None, :] - velocities[None, :, :]
    dist_sq = jnp.sum(diff * diff, axis=-1) + softening**2
    eps = jnp.finfo(positions.dtype).eps
    inv_r = jnp.where(dist_sq > 0, 1.0 / (jnp.sqrt(dist_sq) + eps), 0.0)
    inv_r3 = inv_r / dist_sq
    inv_r5 = inv_r3 / dist_sq
    eye = jnp.eye(positions.shape[0], dtype=bool)
    inv_r3 = jnp.where(eye, 0.0, inv_r3)
    inv_r5 = jnp.where(eye, 0.0, inv_r5)
    rv = jnp.sum(diff * vdiff, axis=-1)
    term = vdiff * inv_r3[..., None] - 3.0 * rv[..., None] * diff * inv_r5[..., None]
    return -G * jnp.sum(masses[None, :, None] * term, axis=1)


def direct_sum_snap(positions, masses, velocities, *, G=1.0, softening=1e-3):
    diff = positions[:, None, :] - positions[None, :, :]
    vdiff = velocities[:, None, :] - velocities[None, :, :]
    dist_sq = jnp.sum(diff * diff, axis=-1) + softening**2
    eps = jnp.finfo(positions.dtype).eps
    inv_r = jnp.where(dist_sq > 0, 1.0 / (jnp.sqrt(dist_sq) + eps), 0.0)
    inv_r3 = inv_r / dist_sq
    inv_r5 = inv_r3 / dist_sq
    inv_r7 = inv_r5 / dist_sq
    eye = jnp.eye(positions.shape[0], dtype=bool)
    inv_r3 = jnp.where(eye, 0.0, inv_r3)
    inv_r5 = jnp.where(eye, 0.0, inv_r5)
    inv_r7 = jnp.where(eye, 0.0, inv_r7)
    rv = jnp.sum(diff * vdiff, axis=-1)
    vv = jnp.sum(vdiff * vdiff, axis=-1)
    term = (
        6.0 * rv[..., None] * vdiff * inv_r5[..., None]
        + 3.0 * vv[..., None] * diff * inv_r5[..., None]
        - 15.0 * (rv * rv)[..., None] * diff * inv_r7[..., None]
    )
    return G * jnp.sum(masses[None, :, None] * term, axis=1)


def direct_sum_crackle(positions, masses, velocities, *, G=1.0, softening=1e-3):
    diff = positions[:, None, :] - positions[None, :, :]
    vdiff = velocities[:, None, :] - velocities[None, :, :]
    dist_sq = jnp.sum(diff * diff, axis=-1) + softening**2
    eps = jnp.finfo(positions.dtype).eps
    inv_r = jnp.where(dist_sq > 0, 1.0 / (jnp.sqrt(dist_sq) + eps), 0.0)
    inv_r3 = inv_r / dist_sq
    inv_r5 = inv_r3 / dist_sq
    inv_r7 = inv_r5 / dist_sq
    inv_r9 = inv_r7 / dist_sq
    eye = jnp.eye(positions.shape[0], dtype=bool)
    inv_r5 = jnp.where(eye, 0.0, inv_r5)
    inv_r7 = jnp.where(eye, 0.0, inv_r7)
    inv_r9 = jnp.where(eye, 0.0, inv_r9)
    rv = jnp.sum(diff * vdiff, axis=-1)
    vv = jnp.sum(vdiff * vdiff, axis=-1)
    term = (
        9.0 * vv[..., None] * vdiff * inv_r5[..., None]
        - 45.0 * (rv * rv)[..., None] * vdiff * inv_r7[..., None]
        - 45.0 * rv[..., None] * vv[..., None] * diff * inv_r7[..., None]
        + 105.0 * (rv * rv * rv)[..., None] * diff * inv_r9[..., None]
    )
    return G * jnp.sum(masses[None, :, None] * term, axis=1)


def relative_l2_error(reference, approx):
    numer = jnp.linalg.norm(approx - reference)
    denom = jnp.maximum(jnp.linalg.norm(reference), jnp.asarray(1e-12, dtype=reference.dtype))
    return numer / denom


In [ ]:
key = jax.random.PRNGKey(7)
key_pos, key_mass, key_vel = jax.random.split(key, 3)
n = 96
softening = 1e-3

positions = jax.random.uniform(key_pos, (n, 3), minval=-1.0, maxval=1.0, dtype=jnp.float32)
masses = jax.random.uniform(key_mass, (n,), minval=0.5, maxval=1.5, dtype=jnp.float32)
velocities = jax.random.uniform(key_vel, (n, 3), minval=-0.2, maxval=0.2, dtype=jnp.float32)

solver = FastMultipoleMethod(
    preset=FMMPreset.FAST,
    basis="solidfmm",
    softening=softening,
)


In [ ]:
acc, (jerk_fmm, snap_fmm, crackle_fmm) = solver.compute_accelerations_with_time_derivatives(
    positions,
    masses,
    velocities,
    leaf_size=16,
    max_order=4,
    max_time_derivative_order=3,
    mode="accurate",
)

jerk_ref = direct_sum_jerk(positions, masses, velocities, softening=softening)
snap_ref = direct_sum_snap(positions, masses, velocities, softening=softening)
crackle_ref = direct_sum_crackle(positions, masses, velocities, softening=softening)


In [ ]:
metrics = {
    "jerk_rel_l2": float(relative_l2_error(jerk_ref, jerk_fmm)),
    "snap_rel_l2": float(relative_l2_error(snap_ref, snap_fmm)),
    "crackle_rel_l2": float(relative_l2_error(crackle_ref, crackle_fmm)),
    "jerk_max_abs": float(jnp.max(jnp.abs(jerk_fmm - jerk_ref))),
    "snap_max_abs": float(jnp.max(jnp.abs(snap_fmm - snap_ref))),
    "crackle_max_abs": float(jnp.max(jnp.abs(crackle_fmm - crackle_ref))),
}
metrics


## Order Sweep

A quick sweep over multipole order helps show the expected accuracy trend.

In [ ]:
def run_order(order):
    _, (jerk, snap, crackle) = solver.compute_accelerations_with_time_derivatives(
        positions,
        masses,
        velocities,
        leaf_size=16,
        max_order=order,
        max_time_derivative_order=3,
        mode="accurate",
    )
    return {
        "max_order": order,
        "jerk_rel_l2": float(relative_l2_error(jerk_ref, jerk)),
        "snap_rel_l2": float(relative_l2_error(snap_ref, snap)),
        "crackle_rel_l2": float(relative_l2_error(crackle_ref, crackle)),
    }

order_sweep = [run_order(order) for order in (2, 3, 4, 5)]
order_sweep


These values are meant as a practical regression/sanity check. For tighter error targets, sweep over `theta`, `max_order`, and particle distributions relevant to your workload.